In [2]:
import pandas as pd

print(pd.__version__)

3.0.5


In [3]:
df = pd.read_csv("dirty_school_data.csv")
df

,نام,کد_ملی,تاریخ_تولد_میلادی,پایه_تحصیلی,رشته_تحصیلی,وزن,قد,معدل,شماره_تلفن,شهریه
0,سجاد کریمی,1463031617,2006-08-14,نهم,علوم تجربی,63.9 lb,184,13.47,09842524105,"$15,000,000"
1,NaN,6688937346,12-May-10,دهم,ادبیات و علوم انسانی,87,1.79,NaN,0999 016 2720,25000000
2,ترانه جعفری,4589297500,2011-02-26,هشتم,ادبیات و علوم انسانی,84 kg,167,11.32,09424154152,"25,000,000"
3,زهرا نجفی,2048996423,1279756800,دهم,فنی حرفه‌ای,145.5 lb,194,نامشخص,00989016668700,"$5,000,000"
4,نگار نجفی,5614978403,2012-09-20,7,فنی حرفه‌ای,66,189,13.28,ندارد,"$75,000,000"
...,...,...,...,...,...,...,...,...,...,...
99,رضا احمدی,9632595327,1301270400,یازدهم,تجربی,30 kg,1.65,نامشخص,0939 500 4797,45000000
100,فاطمه صادقی,284395995,18/02/2008,پایه دهم,فنی حرفه‌ای,54 kg,1.55,21.91,NaN,2000000
101,مریم بهرامی,6805891135,2012-11-14,دهم,علوم انسانی,83 kg,146,15.19,00989993756798,NaN
102,سجاد کاظمی,7676772190,2008-06-22,هفتم,کاردانش,56kg,150 cm,17.73,ندارد,0 تومان


In [ ]:
#cleaning name

# written by reza & baayram

# تمیز کاری ستون نام

invalid_text_values = ['', 'ندارد', 'NAN', 'nan', 'None', 'نامشخص']
df.shape
df.drop_duplicates(inplace=True)
df.shape
df['وضعیت'] = 'معتبر'

def is_invalid_name(value):
    name_str_value = str(value).strip()
    
    if name_str_value in invalid_text_values:
        return True
        
    return False

invalid_name = df['نام'].apply(is_invalid_name)

df.loc[invalid_name, 'وضعیت'] = 'نامعتبر'
df.loc[invalid_name, 'نام'] = 'مهمان'



# تمیز کاری ستون کد ملی

def is_valid(code):
    code = str(code).strip()
    if not (len(code) == 10 and code.isdigit()):
        return False
    digits = [int(n) for n in code]
    total = sum(digits[i] * (10 - i) for i in range(9))
    remainder = total % 11
    control_digit = digits[9]
    return control_digit == (remainder if remainder < 2 else 11 - remainder)

invalid_code = ~df['کد_ملی'].apply(is_valid)
df.loc[invalid_code, 'وضعیت'] = 'نامعتبر'


#تمیز کاری پایه تحصیلی

grades = [
    "اول","دوم","سوم","چهارم","پنجم","ششم","هفتم","هشتم","نهم","دهم","یازدهم","دوازدهم"
]



def clean_grade(value):
    value = value.replace("پایه", "")
    

    number_to_grade = {"1": "اول","2": "دوم","3": "سوم","4": "چهارم","5": "پنجم","6": "ششم","7": "هفتم","8": "هشتم","9": "نهم",
        "10": "دهم", "11": "یازدهم","12": "دوازدهم"
    }

    if pd.isna(value):
        return 0

    if value in number_to_grade:
        return number_to_grade[value]

    if value in grades:
            return value

    else:
        return 0

df["پایه_تحصیلی"] = df["پایه_تحصیلی"].apply(clean_grade)

#تمیز کاری ستون رشته تحصیلی
maygors = [
    "علوم تجربی", "ادبیات و علوم انسانی", "فنی حرفه‌ای", 
    "ریاضی فیزیک", "ریاضی", "علوم انسانی", "تجربی"
]

def clean_maygor(value):
    if pd.isna(value):
        return "نامشخص"

    val = str(value).strip()

    if val in maygors:
        return val
    
    mapping = {
        "تجربی": "علوم تجربی",
        "ریاضی": "ریاضی فیزیک",
        "انسانی": "ادبیات و علوم انسانی"
    }
    
    return mapping.get(val, "نامشخص")

df["رشته_تحصیلی"] = df["رشته_تحصیلی"].apply(clean_maygor)

#تمیز کاری ستون معدل

def is_invalid_score(value):
    str_val = str(value).strip()
    
    if str_val in invalid_text_values:
        return True
    
    try:
        num_val = float(str_val)
        if num_val > 20 or num_val < 0:
            return True
    except ValueError:
        return True 
        
    return False

invalid_score = df['معدل'].apply(is_invalid_score)

df.loc[invalid_score, 'وضعیت'] = 'نامعتبر'
df.loc[invalid_score, 'معدل'] = '0'

df['معدل'] = df['معدل'].astype(float)

#تمیز کاری ستون شهریه

USD_TO_IRR = 195000
df['شهریه'] = df['شهریه'].astype(str).str.strip()


def clean_salary(value):
    str_value = str(value).strip()

    if str_value in invalid_text_values:
        return 0

    try:
        clean_value = (
            str_value
            .replace(',', '')
            .replace('،', '')
            .strip()
        )

        if '$' in clean_value:
            clean_value = clean_value.replace('$', '').strip()
            dollar_value = float(clean_value)
            return dollar_value * USD_TO_IRR
        
        elif 'تومان' in clean_value:
            clean_value = clean_value.replace('تومان', '').strip()
            toman_value = float(clean_value)
            return toman_value * 10

        else:
            return float(clean_value)

    except (ValueError, TypeError):
        return 0


df['شهریه'] = df['شهریه'].apply(clean_salary)
df['شهریه'] = df['شهریه'].round().astype(int)



#تمیز کاری ستون وزن

lb_to_kg = 0.453592
df['وزن'] = df['وزن'].astype(str).str.strip()

def clean_weight(value):
    str_value = str(value).strip()

    if str_value in invalid_text_values:
        return 0

    try:
        clean_value = (
            str_value
            .replace(',', '')
            .replace('،', '')
            .strip()
        )

        if 'lb' in clean_value:
            clean_value = clean_value.replace('lb', '').strip()
            lb_value = float(clean_value)
            return lb_value * lb_to_kg
        
        elif 'pound' in clean_value:
            clean_value = clean_value.replace('pound', '').strip()
            pound_value = float(clean_value)
            return pound_value * lb_to_kg
        
        elif 'kg' in clean_value:
            clean_value = clean_value.replace('kg', '').strip()
            kg_value = float(clean_value)
            return kg_value

        else:
            return float(clean_value)

    except (ValueError, TypeError):
        return 0


df['وزن'] = df['وزن'].apply(clean_weight)
df['وزن'] = df['وزن'].round().astype(int)

# تمیز کاری ستون قد
def clean_height(value):
    if pd.isna(value):
        return 0
    

    val = str(value).lower().replace("cm", "").replace("m", "").strip()
    
    num = float(val)
        
    
    if num < 3:
            return int(num * 100)
    else:
            return int(num)
        

df['قد'] = df['قد'].apply(clean_height)

# تمیز کاری ستون تاریخ
def clean_date(value):

    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    if value in invalid_text_values:
        return pd.NaT

    if value.isdigit():
        return pd.to_datetime(
            int(value),
            unit='s',
            errors='coerce'
        )

    return pd.to_datetime(
        value,
        dayfirst=True,
        errors='coerce'
    )


df['تاریخ_تولد_میلادی'] = df['تاریخ_تولد_میلادی'].apply(clean_date)

# تمیز کاری شماره تلفن
def clean_num(value):
    str_num = str(value).strip()
    
    if str_num in invalid_text_values:
        return 0

    try:
        clean_value = (
            str_num
            .replace(',', '')
            .replace('،', '')
            .replace(' ', '')
            .replace('-', '')
            .replace('0098', '0')
            .strip()
        )

        if not len(clean_value) == 11:
            clean_value = '0'
            return clean_value

        else:
            return clean_value

    except (ValueError, TypeError):
        return 0

df['شماره_تلفن'] = df['شماره_تلفن'].apply(clean_num)




# اشتباهات

"""
Logical Inconsistency: برگرداندن انواع مختلف خروجی از یک تابع (گاهی رشته، گاهی عدد، گاهی بولین).
The None Trap: فراموش کردن return برای حالت‌هایی که شرط if برقرار نیست (باعث حذف شدن داده‌های سالم می‌شد).
The “Inverted Logic” in loc: انتخاب ردیف‌های صحیح به جای ردیف‌های خطا در هنگام جایگزینی.
Mixing Data Types: انجام عملیات ریاضی روی داده‌های رشته‌ای (String) بدون پاکسازی کامل.
Missing else in apply: وقتی از apply استفاده می‌کنی، تابع تو باید برای هر ردیف، حتماً یک خروجی (حتی اگر همان مقدار اصلی باشد) برگرداند.
تبدیل فرمت شماره تلفن: در موارد مشابه به این
9.89086E+11
"""
# df["نام"] =df["نام"].astype(str).str.strip()
# def NameValidate(value):
#     if value == "NAN" or value == "":
#        clean_value = value.replace("NAN","مهمان").replace("","مهمان")
#        return clean_value
#     return value

# df['نام'] = df['نام'].apply(NameValidate)
# df['نام']

# df.loc[df['نام'] == '', 'نام'] = 'Guest'
# df
# df['نام'].dropna()
# df.shape
# df
# df['کد_ملی'] = df['کد_ملی'].astype(str).str.strip()
# def is_valid_national_code(code):
#     if not (len(code) == 10 and code.isdigit()):
#         return '1111111111'

#     digits = [int(d) for d in code]
#     control_digit = digits[9]
    
#     total = sum(digits[i] * (10 - i) for i in range(9))
#     remainder = total % 11
    
#     if remainder < 2:
#         return control_digit == remainder
#     else:
#         return control_digit == (11 - remainder)

# df['کد_ملی'] = df['کد_ملی'].apply(is_valid_national_code)
df.head(50)




C:\Users\hp\AppData\Local\Temp\ipykernel_13132\1360994460.py:246: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(
C:\Users\hp\AppData\Local\Temp\ipykernel_13132\1360994460.py:246: UserWarning: Parsing dates in %Y.%m.%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(
C:\Users\hp\AppData\Local\Temp\ipykernel_13132\1360994460.py:246: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(


,نام,کد_ملی,تاریخ_تولد_میلادی,پایه_تحصیلی,رشته_تحصیلی,وزن,قد,معدل,شماره_تلفن,شهریه,وضعیت
0,سجاد کریمی,1463031617,2006-08-14,نهم,علوم تجربی,29,184,13.47,09842524105,2925000000000,نامعتبر
1,مهمان,6688937346,2010-05-12,0,ادبیات و علوم انسانی,87,179,0.00,09990162720,25000000,نامعتبر
2,ترانه جعفری,4589297500,2011-02-26,هشتم,ادبیات و علوم انسانی,84,167,11.32,09424154152,25000000,نامعتبر
3,زهرا نجفی,2048996423,2010-07-22,دهم,فنی حرفه‌ای,66,194,0.00,09016668700,975000000000,نامعتبر
4,نگار نجفی,5614978403,2012-09-20,هفتم,فنی حرفه‌ای,66,189,13.28,0,14625000000000,نامعتبر
5,مریم نجفی,3040785932,2008-09-14,هفتم,فنی حرفه‌ای,77,0,12.37,09559059929,5850000000000,نامعتبر
6,مهمان,974395339,2007-07-15,هشتم,نامشخص,33,168,11.31,0,2925000000000,نامعتبر
7,فاطمه صادقی,284395995,2008-02-18,0,فنی حرفه‌ای,54,155,0.00,0,2000000,نامعتبر
8,پریسا رضایی,792524191,2011-09-16,هفتم,ریاضی فیزیک,59,163,19.31,9.89086E+11,11700000000000,نامعتبر
9,رها حسینی,2748467737,2010-04-21,0,ادبیات و علوم انسانی,36,190,17.88,0,30000000,نامعتبر
